In [ ]:
from graphviz import Digraph

def trace(root):
    # builds a set of all nodes and edges in a graph
    nodes, edges = set(), set()

    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)

    build(root)
    return nodes, edges


def draw_dot(root):
    dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'})  # LR = left to right

    nodes, edges = trace(root)

    for n in nodes:
        uid = str(id(n))
        # for any value in the graph, create a rectangular ('record') node for it
        dot.node(
            name=uid,
            label="{ data %.4f | grad %.4f }" % (n.data, n.grad),
            shape='record'
        )

        if n._op:
            # if this value is a result of some operation, create an op node for it
            dot.node(name=uid + n._op, label=n._op)
            # and connect this node to it
            dot.edge(uid + n._op, uid)

    for n1, n2 in edges:
        # connect n1 to the op node of n2
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)

    return dot

In [ ]:
# @title
class Value:
    def __init__(self, data, children=(), _op=''):
        self.data = data
        self._prev = set(children)
        self._op = _op
        self.grad = 0.0
        self._backward = lambda: None

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward
        return out
    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)

    def __radd__(self, other):
        return self + other

    def __rmul__(self, other):
        return self * other

    def __rsub__(self, other):
        return other + (-self)

    def __pow__(self, other):
        out = Value(self.data ** other, (self, ), f"**{other}")

        def _backward():
            self.grad += other * (self.data ** (other - 1)) * out.grad
        out._backward = _backward
        return out
    def __repr__(self):
        return f"{self.data}"
    def backward(self):
        visited = set()
        topo = []

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for u in v._prev:
                    build_topo(u)
                topo.append(v)

        build_topo(self)
        self.grad = 1

        for v in reversed(topo):
            v._backward()


In [ ]:
import random

class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x):
        cur = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return cur

    def parameters(self):
        return self.w + [self.b]


class Layer:
    def __init__(self, nin, nout):
        self.layer = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.layer]
        return outs

    def parameters(self):
        return [p for n in self.layer for p in n.parameters()]


class MLP:
    def __init__(self, nin, nouts):
        cur = [nin] + nouts
        self.layers = [Layer(cur[i], cur[i + 1]) for i in range(len(nouts))]

    def __call__(self, x):
        for n in self.layers:
            x = n(x)
        return x

    def parameters(self):
        return [p for n in self.layers for p in n.parameters()]

In [ ]:
xs = [
 [2.0, 3.0, -1.0],
 [3.0, -1.0, 0.5],
 [0.5, 1.0, 1.0],
 [1.0, 1.0, -1.0],
]
n = MLP(3, [4, 4, 1])

In [ ]:
ys = [1.0, -1.0, -1.0, 1.0]

In [ ]:
ypred = [n(x) for x in xs]
loss = sum((cur - want) ** 2 for cur, want in zip(ypred, ys))
loss.backward()
loss.data

2151.551820082417

In [ ]:
for p in n.parameters():
    p.data += -0.05 * p.grad
for p in n.parameters():
    p.grad = 0
loss.backward()
loss.data


2151.551820082417

In [ ]:
loss = sum((cur - want) ** 2 for cur, want in zip(ypred, ys))
loss.backward()
loss.data


4.656856594591042